In [1]:
import pandas as pd
import numpy as np

# STRING

In [3]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

string = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.links.detailed.v12.0.txt',sep=' ')
string_info = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/string_data/9606.protein.info.v12.0.txt',sep='\t')

string['protein1'] = string['protein1'].str.replace('9606.', '')
string['protein2'] = string['protein2'].str.replace('9606.', '')
string_info['#string_protein_id'] = string_info['#string_protein_id'].str.replace('9606.', '')
string_info = string_info.iloc[:, :2]


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2181448581.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')


In [4]:
string = string.merge(string_info, left_on='protein1', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_1'})
string = string.drop('#string_protein_id', axis=1)

# Repeat for 'protein2' column
string = string.merge(string_info, left_on='protein2', right_on='#string_protein_id', how='left')
string = string.rename(columns={'preferred_name': 'gene_symbol_2'})
string = string.drop('#string_protein_id', axis=1)


string


,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2
0,ENSP00000000233,ENSP00000356607,0,0,0,45,134,0,81,173,ARF5,RALGPS2
1,ENSP00000000233,ENSP00000427567,0,0,0,0,128,0,70,154,ARF5,FHDC1
2,ENSP00000000233,ENSP00000253413,0,0,0,118,49,0,69,151,ARF5,ATP6V1E1
3,ENSP00000000233,ENSP00000493357,0,0,0,56,53,0,457,471,ARF5,CYTH2
4,ENSP00000000233,ENSP00000324127,0,0,0,0,46,0,197,201,ARF5,PSD3
...,...,...,...,...,...,...,...,...,...,...,...,...
13715399,ENSP00000501317,ENSP00000475489,0,0,0,60,99,0,126,195,RFX7,MPHOSPH9
13715400,ENSP00000501317,ENSP00000370447,0,0,0,55,111,0,79,158,RFX7,VCX
13715401,ENSP00000501317,ENSP00000312272,0,0,0,0,0,0,227,226,RFX7,YPEL2
13715402,ENSP00000501317,ENSP00000402092,0,0,0,0,67,0,146,169,RFX7,SAMD3


In [5]:
# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan, 'NA'))


# Ensure all entries are treated as strings
string['gene_symbol_1'] = string['gene_symbol_1'].astype(str)
string['gene_symbol_2'] = string['gene_symbol_2'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
string['hgnc_id_1'], string['ensembl_gene_id_1'] = zip(*string['gene_symbol_1'].apply(lookup_gene_info))
string['hgnc_id_2'], string['ensembl_gene_id_2'] = zip(*string['gene_symbol_2'].apply(lookup_gene_info))
string

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,gene_symbol_1,gene_symbol_2,hgnc_id_1,ensembl_gene_id_1,hgnc_id_2,ensembl_gene_id_2
0,ENSP00000000233,ENSP00000356607,0,0,0,45,134,0,81,173,ARF5,RALGPS2,HGNC:658,ENSG00000004059,HGNC:30279,ENSG00000116191
1,ENSP00000000233,ENSP00000427567,0,0,0,0,128,0,70,154,ARF5,FHDC1,HGNC:658,ENSG00000004059,HGNC:29363,ENSG00000137460
2,ENSP00000000233,ENSP00000253413,0,0,0,118,49,0,69,151,ARF5,ATP6V1E1,HGNC:658,ENSG00000004059,HGNC:857,ENSG00000131100
3,ENSP00000000233,ENSP00000493357,0,0,0,56,53,0,457,471,ARF5,CYTH2,HGNC:658,ENSG00000004059,HGNC:9502,ENSG00000105443
4,ENSP00000000233,ENSP00000324127,0,0,0,0,46,0,197,201,ARF5,PSD3,HGNC:658,ENSG00000004059,HGNC:19093,ENSG00000156011
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13715399,ENSP00000501317,ENSP00000475489,0,0,0,60,99,0,126,195,RFX7,MPHOSPH9,HGNC:25777,ENSG00000181827,HGNC:7215,ENSG00000051825
13715400,ENSP00000501317,ENSP00000370447,0,0,0,55,111,0,79,158,RFX7,VCX,HGNC:25777,ENSG00000181827,HGNC:12667,ENSG00000182583
13715401,ENSP00000501317,ENSP00000312272,0,0,0,0,0,0,227,226,RFX7,YPEL2,HGNC:25777,ENSG00000181827,HGNC:18326,ENSG00000175155
13715402,ENSP00000501317,ENSP00000402092,0,0,0,0,67,0,146,169,RFX7,SAMD3,HGNC:25777,ENSG00000181827,HGNC:21574,ENSG00000164483


In [6]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
main_csv

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1094071010.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}
# Also add the reverse pairs since interactions might not be directional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

def get_combined_score(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
    pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # Reverse order

    return string_dict.get(pair1) or string_dict.get(pair2)  # Check both ways

# Apply function to find the interaction score
main_csv['StringInteractionWithBiomarker'] = main_csv.apply(get_combined_score, axis=1)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,985.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,960.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,889.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,606.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
string_biomarker_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_biomarker','StringInteractionWithBiomarker']]
string_biomarker_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_biomarker.csv',index=False)

In [9]:
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}
# Also add the reverse pairs since interactions might not be directional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

def get_combined_score(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
    pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # Reverse order

    return string_dict.get(pair1) or string_dict.get(pair2)  # Check both ways

# Apply function to find the interaction score
main_csv['StringInteractionWithTarget'] = main_csv.apply(get_combined_score, axis=1)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,985.0,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,960.0,964.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,889.0,510.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,606.0,183.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
string_target1_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target1','StringInteractionWithTarget']]
string_target1_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_target1.csv',index=False)

In [11]:
import pandas as pd

# Build the interaction dictionary from the STRING DataFrame
string_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): row['combined_score']
    for _, row in string.iterrows()
}

# Add reverse pairs to make the dictionary bidirectional
string_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): row['combined_score']
    for _, row in string.iterrows()
})

# Define function to get combined interaction score
def get_combined_score(row):
    target2 = row['ensembl_gene_id_target2']

    # Return 0 if target2 is missing or empty
    if pd.isna(target2) or target2 == '':
        return 0

    pair1 = (row['ensembl_gene_id_query'], target2)
    pair2 = (target2, row['ensembl_gene_id_query'])  # Reverse order

    # Return the score if found, else 0
    return string_dict.get(pair1) or string_dict.get(pair2) or 0

# Apply the function to the main DataFrame
main_csv['StringInteractionWithTarget2'] = main_csv.apply(get_combined_score, axis=1)

# Display the updated DataFrame
main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,StringInteractionWithTarget2
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,985.0,999.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,960.0,964.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,889.0,510.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,606.0,183.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,188.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [12]:
string_target2_query = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target2','StringInteractionWithTarget2']]
string_target2_query.to_csv('input_data/3_ML_outputs/feature_output/string_score_query_target2.csv',index=False)

# BIOGRID

In [13]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1717061817.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


In [14]:
hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')
biogrid = biogrid[(biogrid['Organism Name Interactor A'] == 'Homo sapiens') & (biogrid['Organism Name Interactor B'] == 'Homo sapiens')]
biogrid

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2034133230.py:1: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt',sep='\t')
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/2034133230.py:5: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  biogrid = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/biogrid_data/BIOGRID-MV-Physical-4.4.229.tab3.txt',sep='\t')


,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,TREMBL Accessions Interactor B,REFSEQ Accessions Interactor B,Ontology Term IDs,Ontology Term Names,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,Q59H94,NP_001120959|NP_001449,-,-,-,-,-,-,Homo sapiens,Homo sapiens
1,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,NP_150250|NP_150253|NP_150252|NP_150247|NP_150...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
2,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,NP_644805|NP_003141|NP_001356447|NP_001356443|...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
3,586,375,23163,106870,116775,-,-,ARF1,GGA3,-,...,B7Z456|A8K6M0,NP_619525|NP_001278571|NP_001278570|NP_0011661...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
4,612,377,23647,106872,117174,-,-,ARF3,ARFIP2,-,...,B4DUZ3|B4DXH2|A0A087X1E4,NP_036534|NP_001229785|NP_001229784|NP_001229783,-,-,-,-,-,-,Homo sapiens,Homo sapiens
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
472380,3638570,5566,55755,111553,120873,-,RP11-412P1.1,PRKACA,CDK5RAP2,PKACA,...,Q7Z3M0|B3KVI2|B9EG74,NP_060719|NP_001258968|NP_001011649,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472381,3638571,5566,9659,111553,115017,-,RP4-791M13.2,PRKACA,PDE4DIP,PKACA,...,A0A087WYE4|A0A087WVF8,NP_055459|NP_001337451|NP_001185763|NP_0011857...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472382,3638572,5566,113251,111553,125238,-,PP13296,PRKACA,LARP4,PKACA,...,Q6P4E2|Q96J85|Q8TBL5,NP_001339245|NP_001339244|NP_001339247|NP_0013...,-,-,-,-,-,-,Homo sapiens,Homo sapiens
472383,3638573,5566,2534,111553,108810,-,RP1-66H14.1,PRKACA,FYN,PKACA,...,-,NP_002028|NP_001357458|NP_694593|NP_694592,-,-,-,-,-,-,Homo sapiens,Homo sapiens


In [15]:

# Sort values in each row and create a new column containing sorted combinations
biogrid['SortedInteractors'] = biogrid.apply(lambda row: '-'.join(np.sort([row['Official Symbol Interactor A'], row['Official Symbol Interactor B']])), axis=1)

# Drop duplicates based on the new column
biogrid.drop_duplicates(subset='SortedInteractors', keep='first', inplace=True)

# Drop the auxiliary column
biogrid.drop(columns='SortedInteractors', inplace=True)

biogrid= biogrid.reset_index(drop=True)

# Build lookup dictionary
gene_lookup = {}

# Helper function to normalize symbols
def normalize_symbol(s):
    return s.strip().upper() if isinstance(s, str) else None

for entry in hgnc_dict:
    hgnc_id = entry.get('hgnc_id', np.nan)
    ensembl_id = entry.get('ensembl_gene_id', np.nan)

    # Primary symbol
    symbol = normalize_symbol(entry.get('symbol'))
    if symbol:
        gene_lookup[symbol] = (hgnc_id, ensembl_id)

    # Previous symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('prev_symbol')):
        prev_symbols = entry['prev_symbol'].split('|')
        for prev_symbol in prev_symbols:
            norm_prev = normalize_symbol(prev_symbol)
            if norm_prev and norm_prev not in gene_lookup:
                gene_lookup[norm_prev] = (hgnc_id, ensembl_id)

    # Alias symbols (only add if not already in gene_lookup)
    if pd.notnull(entry.get('alias_symbol')):
        alias_symbols = entry['alias_symbol'].split('|')
        for alias in alias_symbols:
            norm_alias = normalize_symbol(alias)
            if norm_alias and norm_alias not in gene_lookup:
                gene_lookup[norm_alias] = (hgnc_id, ensembl_id)

# Lookup function using normalized input
def lookup_gene_info(gene_symbol):
    key = normalize_symbol(gene_symbol)
    return gene_lookup.get(key, (np.nan, np.nan))



# Ensure all entries are treated as strings
biogrid['Official Symbol Interactor A'] = biogrid['Official Symbol Interactor A'].astype(str)
biogrid['Official Symbol Interactor B'] = biogrid['Official Symbol Interactor B'].astype(str)

# Apply the lookup function to both gene_symbol_1 and gene_symbol_2 columns
biogrid['hgnc_id_1'], biogrid['ensembl_gene_id_1'] = zip(*biogrid['Official Symbol Interactor A'].apply(lookup_gene_info))
biogrid['hgnc_id_2'], biogrid['ensembl_gene_id_2'] = zip(*biogrid['Official Symbol Interactor B'].apply(lookup_gene_info))

biogrid

,#BioGRID Interaction ID,Entrez Gene Interactor A,Entrez Gene Interactor B,BioGRID ID Interactor A,BioGRID ID Interactor B,Systematic Name Interactor A,Systematic Name Interactor B,Official Symbol Interactor A,Official Symbol Interactor B,Synonyms Interactor A,...,Ontology Term Categories,Ontology Term Qualifier IDs,Ontology Term Qualifier Names,Ontology Term Types,Organism Name Interactor A,Organism Name Interactor B,hgnc_id_1,ensembl_gene_id_1,hgnc_id_2,ensembl_gene_id_2
0,103,6416,2318,112315,108607,-,-,MAP2K4,FLNC,JNKK|JNKK1|MAPKK4|MEK4|MKK4|PRKMK4|SAPKK-1|SAP...,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:6844,ENSG00000065559,HGNC:3756,ENSG00000128591
1,278,2624,5371,108894,111384,-,-,GATA2,PML,DCML|IMD21|MONOMAC|NFE1B,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:4171,ENSG00000179348,HGNC:9113,ENSG00000140464
2,418,6118,6774,112038,112651,RP4-547C9.3,-,RPA2,STAT3,REPA2|RP-A p32|RP-A p34|RPA32,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:10290,ENSG00000117748,HGNC:11364,ENSG00000168610
3,586,375,23163,106870,116775,-,-,ARF1,GGA3,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:652,ENSG00000143761,HGNC:17079,ENSG00000125447
4,612,377,23647,106872,117174,-,-,ARF3,ARFIP2,-,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:654,ENSG00000134287,HGNC:17160,ENSG00000132254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84513,3629930,582,9867,107058,115200,-,-,BBS1,PJA2,BBS2L2,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:966,ENSG00000174483,HGNC:17481,ENSG00000198961
84514,3629941,8975,7291,114465,113142,-,-,USP13,TWIST1,ISOT3|IsoT-3,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:12611,ENSG00000058056,HGNC:12428,ENSG00000122691
84515,3629949,5562,1317,111549,107712,-,RP11-110L1.1,PRKAA1,SLC31A1,AMPK|AMPKa1,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9376,ENSG00000132356,HGNC:11016,ENSG00000136868
84516,3638506,5566,2534,111553,108810,-,RP1-66H14.1,PRKACA,FYN,PKACA,...,-,-,-,-,Homo sapiens,Homo sapiens,HGNC:9380,ENSG00000072062,HGNC:4037,ENSG00000010810


In [16]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_biomarker'])
    pair2 = (row['ensembl_gene_id_biomarker'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryBiomarker'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [17]:
biogrid_query_biomarker = main_csv[['ensembl_gene_id_query','ensembl_gene_id_biomarker','BIOGRIDPhysicalInteractionQueryBiomarker']]
biogrid_query_biomarker.to_csv('input_data/3_ML_outputs/feature_output/biogrid_biomarker_query.csv',index=False)
biogrid_query_biomarker

,ensembl_gene_id_query,ensembl_gene_id_biomarker,BIOGRIDPhysicalInteractionQueryBiomarker
0,ENSG00000105647,ENSG00000171862,1
1,ENSG00000165458,ENSG00000171862,0
2,ENSG00000103197,ENSG00000171862,0
3,ENSG00000186575,ENSG00000171862,0
4,ENSG00000018610,ENSG00000171862,0
...,...,...,...
144075,ENSG00000203995,ENSG00000213281,0
144076,ENSG00000162378,ENSG00000213281,0
144077,ENSG00000159840,ENSG00000213281,0
144078,ENSG00000074755,ENSG00000213281,0


In [18]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target1'])
    pair2 = (row['ensembl_gene_id_target1'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryTarget1'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0


In [19]:
# Create a dictionary to store interactions (ignoring scores, just existence)
biogrid_dict = {
    (row['ensembl_gene_id_1'], row['ensembl_gene_id_2']): 1
    for _, row in biogrid.iterrows()
}
# Add the reverse pairs to ensure undirected interactions
biogrid_dict.update({
    (row['ensembl_gene_id_2'], row['ensembl_gene_id_1']): 1
    for _, row in biogrid.iterrows()
})

def check_biogrid_interaction(row):
    pair1 = (row['ensembl_gene_id_query'], row['ensembl_gene_id_target2'])
    pair2 = (row['ensembl_gene_id_target2'], row['ensembl_gene_id_query'])  # Reverse order

    return 1 if pair1 in biogrid_dict or pair2 in biogrid_dict else 0

# Apply function to find the interaction existence (1 or 0)
main_csv['BIOGRIDPhysicalInteractionQueryTarget2'] = main_csv.apply(check_biogrid_interaction, axis=1)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1,BIOGRIDPhysicalInteractionQueryTarget2
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [20]:
main_csv['BIOGRIDPhysicalInteractionQueryTarget'] = (main_csv['BIOGRIDPhysicalInteractionQueryTarget1'] | main_csv['BIOGRIDPhysicalInteractionQueryTarget2']).astype(int)

main_csv


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,BIOGRIDPhysicalInteractionQueryBiomarker,BIOGRIDPhysicalInteractionQueryTarget1,BIOGRIDPhysicalInteractionQueryTarget2,BIOGRIDPhysicalInteractionQueryTarget
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0


In [21]:
biogrid_query_target = main_csv[['ensembl_gene_id_query','ensembl_gene_id_target1','BIOGRIDPhysicalInteractionQueryTarget']]
biogrid_query_target.to_csv('input_data/3_ML_outputs/feature_output/biogrid_target_query.csv',index=False)
biogrid_query_target

,ensembl_gene_id_query,ensembl_gene_id_target1,BIOGRIDPhysicalInteractionQueryTarget
0,ENSG00000105647,ENSG00000051382,1
1,ENSG00000165458,ENSG00000051382,0
2,ENSG00000103197,ENSG00000051382,0
3,ENSG00000186575,ENSG00000051382,0
4,ENSG00000018610,ENSG00000051382,0
...,...,...,...
144075,ENSG00000203995,ENSG00000169032,0
144076,ENSG00000162378,ENSG00000169032,0
144077,ENSG00000159840,ENSG00000169032,0
144078,ENSG00000074755,ENSG00000169032,0


# BIOMARKER TSG or Oncogenes

In [22]:
main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')
main_csv

/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_98726/1094071010.py:1: DtypeWarning: Columns (5,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  main_csv = pd.read_csv('input_data/3_ML_outputs/datasets/PredictingSLResistanceFeatures_main.csv')


,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,Class,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,Not_Resistant,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
main_csv['Biomarker'].unique()

array(['PTEN', 'NRAS', 'BRAF', 'ARID1A', 'KRAS', 'BRCA1'], dtype=object)

In [24]:
cancer_gene_census = pd.read_csv('/Users/metinyazar/Desktop/drug_resistance_ppi/new_analysis_with_hit_selection/predicting_resistance/input_data/Cancer_gene_census_data.csv')
cancer_gene_census

,Gene Symbol,Name,Entrez GeneId,Genome Location,Tier,Hallmark,Chr Band,Somatic,Germline,Tumour Types(Somatic),Tumour Types(Germline),Cancer Syndrome,Tissue Type,Molecular Genetics,Role in Cancer,Mutation Types,Translocation Partner,Other Germline Mut,Other Syndrome,Synonyms
0,A1CF,APOBEC1 complementation factor,29974.0,10:50799421-50885675,2,NaN,10q11.23,yes,NaN,melanoma,NaN,NaN,E,NaN,oncogene,Mis,NaN,NaN,NaN,"ACF,ACF64,ACF65,APOBEC1CF,ASP,CCDS73133.1,ENSG..."
1,ABI1,abl-interactor 1,10006.0,10:26746593-26860935,1,Yes,10p12.1,yes,NaN,AML,NaN,NaN,L,Dom,"TSG, fusion",T,KMT2A,NaN,NaN,"ABI-1,CCDS7150.1,E3B1,ENSG00000136754.17,NM_00..."
2,ABL1,v-abl Abelson murine leukemia viral oncogene h...,25.0,9:130713946-130885683,1,Yes,9q34.12,yes,NaN,"CML, ALL, T-ALL",NaN,NaN,L,Dom,"oncogene, fusion","T, Mis","BCR, ETV6, NUP214",NaN,NaN,"ABL,CCDS35165.1,ENSG00000097007.17,JTK7,NM_007..."
3,ABL2,"c-abl oncogene 2, non-receptor tyrosine kinase",27.0,1:179099327-179229601,1,NaN,1q25.2,yes,NaN,AML,NaN,NaN,L,Dom,"oncogene, fusion",T,ETV6,NaN,NaN,"ABLL,ARG,CCDS30947.1,ENSG00000143322.19,NM_007..."
4,ACKR3,atypical chemokine receptor 3,57007.0,2:236569641-236582358,1,Yes,2q37.3,yes,NaN,lipoma,NaN,NaN,M,Dom,"oncogene, fusion",T,HMGA2,NaN,NaN,"CCDS2516.1,CMKOR1,CXCR7,ENSG00000144476.5,GPR1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
748,ZNF429,zinc finger protein 429,353088.0,19:21505564-21538078,2,NaN,19p12,yes,NaN,GBM,NaN,NaN,O,NaN,NaN,Mis,NaN,NaN,NaN,"CCDS42537.1,ENSG00000197013.9,NM_001001415.2,N..."
749,ZNF479,zinc finger protein 479,90827.0,7:57119614-57139864,2,NaN,7p11.2,yes,NaN,"lung cancer, bladder carcinoma, prostate carci...",NaN,NaN,E,NaN,NaN,Mis,NaN,NaN,NaN,"CCDS43590.1,ENSG00000185177.12,KR19,NM_033273...."
750,ZNF521,zinc finger protein 521,25925.0,18:25061926-25352152,1,NaN,18q11.2,yes,NaN,ALL,NaN,NaN,L,Dom,"oncogene, fusion",T,PAX5,NaN,NaN,"CCDS32806.1,EHZF,ENSG00000198795.10,Evi3,NM_01..."
751,ZNRF3,zinc and ring finger 3,84133.0,22:28883592-29057487,2,NaN,22q12.1,yes,NaN,"colorectal cancer, adrenocortical carcinoma, g...",NaN,NaN,E,NaN,TSG,"N, F, Mis",NaN,NaN,NaN,"BK747E2.3,CCDS56225.1,ENSG00000183579.15,FLJ22..."


In [25]:
import pandas as pd
import numpy as np

# First, ensure that the ID columns are the same type (usually integers or strings)
main_csv['Biomarker'] = main_csv['Biomarker'].astype(str)
cancer_gene_census['Gene Symbol'] = cancer_gene_census['Gene Symbol'].astype(str)

# Create a mapping from Entrez GeneId to Role in Cancer
role_mapping = cancer_gene_census.set_index('Gene Symbol')['Role in Cancer'].to_dict()

# Define a function to classify based on role
def classify_gene(entrez_id):
    role = role_mapping.get(entrez_id, None)
    if role:
        if 'TSG' in role:
            return 1
        elif 'oncogene' in role:
            return 0
    return np.nan  # Leave empty if no match or role is unknown

# Apply classification
main_csv['TSG_Label'] = main_csv['Biomarker'].apply(classify_gene)
main_csv

,Screen,SL_Pair,Query,Biomarker,Target1,Target2,entrez_id_query,hgnc_id_query,ensembl_gene_id_query,entrez_id_biomarker,...,StringInteractionWithBiomarker,StringInteractionWithTarget,CoexpressionWithBiomarker,CoexpressionWithTarget,AvgExpression,CoessentialityWithBiomarker,CoessentialityWithTarget,FET_SharedInteractors_Biomarker,FET_SharedInteractors_Target,TSG_Label
0,Dunn,PTEN_PIK3CB,PIK3R2,PTEN,PIK3CB,NaN,5296.0,HGNC:8980,ENSG00000105647,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,Dunn,PTEN_PIK3CB,INPPL1,PTEN,PIK3CB,NaN,3636.0,HGNC:6080,ENSG00000165458,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,Dunn,PTEN_PIK3CB,TSC2,PTEN,PIK3CB,NaN,7249.0,HGNC:12363,ENSG00000103197,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
3,Dunn,PTEN_PIK3CB,NF2,PTEN,PIK3CB,NaN,4771.0,HGNC:7773,ENSG00000186575,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,Dunn,PTEN_PIK3CB,CXorf56,PTEN,PIK3CB,NaN,63932.0,HGNC:26239,ENSG00000018610,5728.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144075,Hayes_Krall,NRAS_MEK,ZYG11A,NRAS,MEK1,MEK2,440590.0,HGNC:32058,ENSG00000203995,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144076,Hayes_Krall,NRAS_MEK,ZYG11B,NRAS,MEK1,MEK2,79699.0,HGNC:25820,ENSG00000162378,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144077,Hayes_Krall,NRAS_MEK,ZYX,NRAS,MEK1,MEK2,7791.0,HGNC:13200,ENSG00000159840,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
144078,Hayes_Krall,NRAS_MEK,ZZEF1,NRAS,MEK1,MEK2,23140.0,HGNC:29027,ENSG00000074755,4893.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [26]:
biomarker_type = main_csv[['Biomarker','TSG_Label']]
biomarker_type

,Biomarker,TSG_Label
0,PTEN,1
1,PTEN,1
2,PTEN,1
3,PTEN,1
4,PTEN,1
...,...,...
144075,NRAS,0
144076,NRAS,0
144077,NRAS,0
144078,NRAS,0


In [ ]:
biomarker_type.to_csv('input_data/3_ML_outputs/feature_output/biomarker_type.csv', index=False)